# Energy Prices vs Inflation Analysis
Determine what kind of correlation does gasoline, crude oil, and natural gas prices havw with inflation.

The following are the steps for analysis:
1. Load datasets
2. Preview each dataframe
3. Clean dates
4. Merge into one dataset
5. Visualize trends
6. Compute correlations
7. My analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


### First I'll read the CSV files and store them in separate dataframes.

In [ ]:
gas = pd.read_csv("../data/raw/gas_prices.csv")
oil = pd.read_csv("../data/raw/crude_oil.csv")
natural = pd.read_csv("../data/raw/national_prices.csv")
inflation = pd.read_csv("../data/raw/inflation.csv")

### Before doing analysis, I'll quickly check the first 5 rows of each dataset.

In [ ]:
print("Gas")
display(gas.head())

print("Oil")
display(oil.head())

print("Natural Gas")
display(natural.head())

print("Inflation")
display(inflation.head())

### Clean dates and keep only needed columns

Convert date columns into datetime format and simplify each dataframe.

In [ ]:
gas["period"] = pd.to_datetime(gas["period"])
oil["period"] = pd.to_datetime(oil["period"])
natural["period"] = pd.to_datetime(natural["period"])
inflation["date"] = pd.to_datetime(inflation["date"])

gas = gas[["period","value"]]
oil = oil[["period","value"]]
natural = natural[["period","value"]]
inflation = inflation[["date","value"]]

gas.columns = ["date","gas"]
oil.columns = ["date","oil"]
natural.columns = ["date","natural_gas"]
inflation.columns = ["date","inflation"]

### I'll join everything together by date so values appear side by side.

Because inflation data is monthly and the others are weekly, I'll make some changes so that all of them shows monthly data.

In [ ]:
# Convert all dates into month-year only
gas["date"] = gas["date"].dt.to_period("M")
oil["date"] = oil["date"].dt.to_period("M")
natural["date"] = natural["date"].dt.to_period("M")
inflation["date"] = inflation["date"].dt.to_period("M")

# Now I have 4-5 of values in the same month. Average all rows inside each of those months
gas = gas.groupby("date")["gas"].mean().reset_index()
oil = oil.groupby("date")["oil"].mean().reset_index()
natural = natural.groupby("date")["natural_gas"].mean().reset_index()

df = gas.merge(oil,on="date")
df = df.merge(natural,on="date")
df = df.merge(inflation,on="date")

display(df.head())

### Plot Trends

I'll use low opacity so that overlapping lines remain visible.

In [ ]:
# Convert date if needed
if str(df["date"].dtype) == "period[M]":
    df["date"] = df["date"].dt.to_timestamp()

# Make numeric
cols = ["gas", "oil", "natural_gas", "inflation"]

for col in cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove missing values created during conversion
df = df.dropna()

# Create copy
scaled = df.copy()

# Normalize
for col in cols:
    scaled[col] = scaled[col] / scaled[col].iloc[0]

# Plot
plt.figure(figsize=(14,6))

for col in cols:
    plt.plot(
        scaled["date"],
        scaled[col],
        label=col,
        alpha=0.6
    )

plt.xlabel("Date")
plt.ylabel("Normalized Value")
plt.title("Normalized Trends: Gas, Oil, Natural Gas, Inflation")
plt.legend()

plt.show()

## Correlation Analysis

Now to calculate how strongly/weakly each variable moves with inflation.

Values close to:

- 1 = strong positive relationship
- -1 = strong negative relationship
- 0 = weak relationship

In [ ]:
corr = df.corr(numeric_only=True)

display(corr)

print(
    corr["inflation"]
    .sort_values(ascending=False)
)